In [1]:
import torch
import torch.nn as nn

In [2]:
x = torch.tensor([
    [-1, -1],
    [-1,  1],
    [ 1, -1],
    [ 1,  1],
], dtype=torch.float32)

y = torch.tensor([
    [0],
    [1],
    [1],
    [0],
], dtype=torch.float32)

In [5]:
class XORMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2,4)
        self.layer2 = nn.Linear(4,1)

    def forward(self,x):
        y_1 = self.layer1(x)
        y_2 = torch.relu(y_1)
        y_3 = self.layer2(y_2)
        return y_3



model = XORMLP()
print (model)
output = model(x)
print(output.shape)

total = 0

for name, parameter in model.named_parameters():
    print(name, parameter.shape, parameter.numel())
    total += parameter.numel()

print("Total parameters:", total)

XORMLP(
  (layer1): Linear(in_features=2, out_features=4, bias=True)
  (layer2): Linear(in_features=4, out_features=1, bias=True)
)
torch.Size([4, 1])
layer1.weight torch.Size([4, 2]) 8
layer1.bias torch.Size([4]) 4
layer2.weight torch.Size([1, 4]) 4
layer2.bias torch.Size([1]) 1
Total parameters: 17


In [6]:
class XORMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2,4)
        self.layer2 = nn.Linear(4,1)

    def forward(self,x):
        y_1 = self.layer1(x)
        y_2 = torch.relu(y_1)
        y_3 = self.layer2(y_2)
        return y_3


model = XORMLP()
loss_api = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(),lr= 0.1)

for step in range(3000):
    optimizer.zero_grad(set_to_none=True)
    y_predict = model(x)
    loss = loss_api(y_predict,y)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        y_predict = model(x)
        loss = loss_api(y_predict,y)
    if (step + 1) % 500 == 0:
        print(f"Step {step + 1}: loss={loss.item():.6f}")
        print("Logits:", y_predict.tolist())
        print("Probabilities:", torch.sigmoid(y_predict))
        print("Predictions:", (torch.sigmoid(y_predict) >= 0.5).long())


Step 500: loss=0.353010
Logits: [[-4.027920246124268], [-0.05229661241173744], [4.901486873626709], [-0.05229661241173744]]
Probabilities: tensor([[0.0175],
        [0.4869],
        [0.9926],
        [0.4869]])
Predictions: tensor([[0],
        [0],
        [1],
        [0]])
Step 1000: loss=0.348473
Logits: [[-5.354372501373291], [-0.004594381898641586], [5.850348472595215], [-0.004594381898641586]]
Probabilities: tensor([[0.0047],
        [0.4989],
        [0.9971],
        [0.4989]])
Predictions: tensor([[0],
        [0],
        [1],
        [0]])
Step 1500: loss=0.347623
Logits: [[-5.976172924041748], [-0.0018915438558906317], [6.398085117340088], [-0.0018915438558906317]]
Probabilities: tensor([[0.0025],
        [0.4995],
        [0.9983],
        [0.4995]])
Predictions: tensor([[0],
        [0],
        [1],
        [0]])
Step 2000: loss=0.347280
Logits: [[-6.394025802612305], [-0.0011010636808350682], [6.763118267059326], [-0.0011010636808350682]]
Probabilities: tensor([[0.001

In [7]:
state = model.state_dict()
for name, tensor in state.items():
    print(name, tensor.shape)

layer1.weight torch.Size([4, 2])
layer1.bias torch.Size([4])
layer2.weight torch.Size([1, 4])
layer2.bias torch.Size([1])


In [8]:
from pathlib import Path
print(Path.cwd())

g:\neural-networks-to-cs336\notebooks\stage_a


In [9]:
Path("checkpoints").mkdir(exist_ok=True)
torch.save(model.state_dict(), "checkpoints/xor_mlp.pt")
state = torch.load(
    "checkpoints/xor_mlp.pt",
    map_location="cpu",
    weights_only=True,
)
loaded_model = XORMLP()
loaded_model.load_state_dict(state)

<All keys matched successfully>

In [10]:
with torch.no_grad():
    y_predict2 = loaded_model(x)
    print(torch.allclose(y_predict,y_predict2))

True
